# 第 12 章：RAG 检索增强生成

这个 notebook 对应 `lessons/12_rag_baseline.md`，演示一个本地可运行的 RAG baseline：chunk、embedding index、top-k retrieval、context prompt、answer + citation，以及无答案拒答路径。

In [ ]:
from src.rag.baseline import (
    Document,
    HashingTextEmbedder,
    VectorStore,
    build_context_prompt,
    chunk_documents,
    run_rag,
    validate_citations,
)

## 1. Documents -> Chunks

chunk 必须保留 `doc_id`、`chunk_id`、title、source 和文本范围，后续 citation 才能追溯。

In [ ]:
documents = [
    Document("legal", "合同", "合同条款要求甲方承担全部责任。乙方应按期支付费用。"),
    Document("medical", "医学", "胸痛和呼吸困难属于危险信号，应及时就医。"),
]
chunks = chunk_documents(documents, chunk_size=24, overlap=4)
for chunk in chunks:
    print(chunk)

## 2. Build Index 与 Top-k Retrieval

这里用确定性的字符 hashing embedding，方便离线测试。真实项目可以替换成 embedding model + vector database。

In [ ]:
store = VectorStore(chunks, HashingTextEmbedder(dim=64))
results = store.search("呼吸困难危险信号", top_k=2)
for result in results:
    print(round(result.score, 4), result.chunk.chunk_id, result.chunk.text)

## 3. Prompt With Context

RAG prompt 要明确要求模型只基于资料回答，资料不足时拒答，并引用来源。

In [ ]:
prompt = build_context_prompt("呼吸困难是不是危险信号？", results)
print(prompt)

## 4. Answer + Citation

教学版 generator 直接基于最高分 chunk 组织答案，并返回 citation。

In [ ]:
answer = run_rag("呼吸困难是不是危险信号？", store, top_k=2, min_score=0.1)
validate_citations(answer, chunks)
print(answer.answer)
print(answer.citations)

## 5. 无答案拒答

当检索分数不足时，系统应走拒答路径，而不是编造答案。

In [ ]:
no_answer = run_rag("量子芯片制造步骤", store, top_k=1, min_score=0.9)
print(no_answer.refused)
print(no_answer.answer)
print(no_answer.citations)